# 第12章：共享内存控制 -- Block Pointer 的威力

## 前置知识
- 第09章：分块矩阵乘法基础
- 第11章：Autotuning
- 理解 GPU 内存层级（Global Memory → L2 Cache → Shared Memory → Register）

## 学习目标
- 理解 **Shared Memory** 在 GEMM 中的核心作用
- 掌握 `tl.make_block_ptr` 的用法 —— 结构化 2D 内存访问
- 理解 `boundary_check` 与手动 mask 的区别
- 实现基于 Block Pointer 的 GEMM kernel
- 理解 Bank Conflict 概念及 Triton 如何自动处理

## 对应 CUDA 代码
- `src/simt/02simt_smem.cu` — 手动管理 Shared Memory 的分块 GEMM
- 在 CUDA 中需要 `__shared__` 声明 + 手动数据搬运 + `__syncthreads()`
- Triton 编译器自动完成这些操作，但 `make_block_ptr` 给予更多控制

In [ ]:
import torch
import triton
import triton.language as tl

## 12.1 Shared Memory 在 GEMM 中的角色

### GPU 内存层级回顾

```
┌──────────────────────────────────────────────────────────────┐
│                    GPU 内存层级                              │
│                                                              │
│  ┌──────────┐   速度最快   容量最小                          │
│  │ Register │   ~TB/s     ~256 KB/SM                        │
│  └────┬─────┘                                                │
│       │                                                      │
│  ┌────▼──────────┐                                           │
│  │ Shared Memory │  ~TB/s    ~48-164 KB/SM                  │
│  │  (on-chip)    │  ◄── 本章重点                             │
│  └────┬──────────┘                                           │
│       │                                                      │
│  ┌────▼─────┐                                                │
│  │ L2 Cache │   ~2-4 TB/s   ~4-40 MB                       │
│  └────┬─────┘                                                │
│       │                                                      │
│  ┌────▼──────────┐                                           │
│  │ Global Memory │  ~1-2 TB/s   ~16-80 GB (HBM)            │
│  │  (off-chip)   │  ◄── 最慢                                │
│  └──────────────┘                                            │
└──────────────────────────────────────────────────────────────┘
```

### 为什么需要 Shared Memory？

在 GEMM 的 K 循环中，每个 tile 的数据需要被多个线程重复使用：

```
不使用 Shared Memory:
  Thread 0: Global → Register (读 A[0,:])  → 计算
  Thread 1: Global → Register (读 A[0,:])  → 计算  ← 重复读取!
  Thread 2: Global → Register (读 A[0,:])  → 计算  ← 重复读取!
  ...
  每次都从 Global Memory (~1TB/s) 读取

使用 Shared Memory:
  协作加载: Global → Shared Memory (一次)  ← 只读一次!
  Thread 0: Shared → Register → 计算
  Thread 1: Shared → Register → 计算       ← 从快速 SMEM 读
  Thread 2: Shared → Register → 计算       ← ~TB/s 带宽
  ...
```

### CUDA vs Triton 的 Shared Memory 管理

```
CUDA (simt_smem.cu):
  __shared__ float4 smem_A[BlockTileM * ldm_blockA_f4size];  // 手动声明
  __shared__ float4 smem_B[BlockTileK * ldm_blockB_f4size];  // 手动声明
  
  // 手动从 Global 搬运到 Shared
  for (int i = tid; i < ...; i += blockDim.x * blockDim.y) {
      float4 buffer = gmem_ptr[...];
      smem_ptr[...] = buffer;
  }
  __syncthreads();  // 手动同步

Triton:
  a_tile = tl.load(a_ptrs, ...)  // 编译器自动决定是否使用 SMEM
  # 或者使用 make_block_ptr 给编译器更多提示
```

**关键洞察**：Triton 编译器根据数据访问模式自动决定何时使用 Shared Memory。
但使用 `tl.make_block_ptr` 可以给编译器提供**结构化的访问模式信息**，
帮助它做出更好的决策。

## 12.2 tl.make_block_ptr — 结构化 2D 内存访问

### 为什么需要 Block Pointer？

在第09章中，我们使用 **手动指针计算** 来访问矩阵 tile：

```python
# 手动方式：构造 2D 指针矩阵
rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)       # (BLOCK_M,)
rk = tl.arange(0, BLOCK_K)                          # (BLOCK_K,)
a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak  # (BLOCK_M, BLOCK_K)

# 加载时需要手动处理边界
a_tile = tl.load(a_ptrs, mask=(rm[:, None] < M) & (rk[None, :] < K), other=0.0)

# 移动到下一个 K tile
a_ptrs += BLOCK_K * stride_ak
```

这种方式的问题：
1. 指针计算冗长，容易出错
2. 编译器难以推断出这是**规则的 2D block 访问**
3. mask 计算增加了额外的指令开销

### make_block_ptr 的优势

```python
# Block Pointer 方式：声明一个结构化的 2D 块指针
a_block_ptr = tl.make_block_ptr(
    base=a_ptr,                              # 矩阵基地址
    shape=(M, K),                            # 整个矩阵的形状
    strides=(stride_am, stride_ak),          # 矩阵的步长
    offsets=(pid_m * BLOCK_M, 0),            # 当前 block 的起始偏移
    block_shape=(BLOCK_M, BLOCK_K),          # 要加载的 block 大小
    order=(1, 0),                            # 内存布局顺序
)

# 加载时自动处理边界
a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))

# 移动到下一个 K tile —— 简洁!
a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
```

### make_block_ptr 参数详解

```
tl.make_block_ptr(
    base,          # 基地址指针 (矩阵首元素的地址)
    shape,         # 整个矩阵的逻辑形状 (M, K) 或 (K, N)
    strides,       # 每个维度的步长 (元素为单位)
    offsets,       # 当前 block 在矩阵中的偏移 (行偏移, 列偏移)
    block_shape,   # 要加载的 block 的形状 (BLOCK_M, BLOCK_K)
    order,         # 内存布局: (1,0) = 行主序, (0,1) = 列主序
)
```

```
矩阵 A (M × K), 行主序:

  列: 0  1  2  3  4  5  6  7  8  9  ... K-1
  ┌─────────────────────────────────────────┐
0 │  .  .  .  .  .  .  .  .  .  .       .  │
1 │  .  .  .  .  .  .  .  .  .  .       .  │
  │     ┌─────────────┐                     │
2 │  .  │ A_tile      │  .  .  .  .      .  │ ← offsets = (2, 1)
3 │  .  │ (BLOCK_M=4, │  .  .  .  .      .  │   block_shape = (4, 3)
4 │  .  │  BLOCK_K=3) │  .  .  .  .      .  │
5 │  .  │             │  .  .  .  .      .  │
  │     └─────────────┘                     │
6 │  .  .  .  .  .  .  .  .  .  .       .  │
  └─────────────────────────────────────────┘

  order = (1, 0) 表示: 维度 1 (列) 在内存中连续 → 行主序
  order = (0, 1) 表示: 维度 0 (行) 在内存中连续 → 列主序
```

## 12.3 boundary_check vs 手动 mask

### 手动 mask 方式（第09章）

```python
mask = (rm[:, None] < M) & (rk[None, :] + k_offset < K)
a_tile = tl.load(a_ptrs, mask=mask, other=0.0)
```

- 需要手动构造 2D boolean mask
- 每次加载都需要计算 mask
- 灵活但冗长

### boundary_check 方式（Block Pointer）

```python
a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
```

- `boundary_check=(0, 1)` 表示在维度 0 (M) 和维度 1 (K) 上检查边界
- 编译器根据 `shape` 和 `offsets` 自动生成最优的边界检查代码
- 越界元素自动填充为 0

```
边界情况示例 (M=5, K=7, BLOCK_M=4, BLOCK_K=4):

最后一个 M 方向的 block (pid_m=1, offset_m=4):
  ┌───┬───┬───┬───┐
  │ x │ x │ x │ x │  ← 行 4 (有效)
  ├───┼───┼───┼───┤
  │ 0 │ 0 │ 0 │ 0 │  ← 行 5 (越界, 填 0)  boundary_check=(0,)
  ├───┼───┼───┼───┤
  │ 0 │ 0 │ 0 │ 0 │  ← 行 6 (越界, 填 0)
  ├───┼───┼───┼───┤
  │ 0 │ 0 │ 0 │ 0 │  ← 行 7 (越界, 填 0)
  └───┴───┴───┴───┘

最后一个 K 方向的 tile (k_offset=4):
  ┌───┬───┬───┬───┐
  │ x │ x │ x │ 0 │  ← 列 4,5,6 有效; 列 7 越界  boundary_check=(1,)
  ├───┼───┼───┼───┤
  │ x │ x │ x │ 0 │
  └───┴───┴───┴───┘

同时在两个维度检查:
  boundary_check=(0, 1)  → 两个维度都检查
```

### 性能差异

- 当矩阵尺寸是 BLOCK 大小的整数倍时，`boundary_check` 基本零开销
- 编译器可以证明不需要检查时会完全优化掉
- 手动 mask 始终需要计算 boolean 矩阵

## 12.4 实现：Block Pointer GEMM Kernel

现在我们用 `make_block_ptr` 重新实现分块 GEMM，对比第09章的手动指针版本。

In [ ]:
@triton.jit
def matmul_block_ptr_kernel(
    # 矩阵指针
    a_ptr, b_ptr, c_ptr,
    # 矩阵维度
    M, N, K,
    # 步长
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # 分块参数 (编译时常量)
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    使用 Block Pointer 的 GEMM kernel。
    
    对应 CUDA simt_smem.cu：
    - 在 CUDA 中需要手动声明 __shared__ float4 smem_A[...], smem_B[...]
    - 手动计算线程到 smem 的映射
    - 手动搬运 global → smem，然后 smem → register
    - Triton 的 make_block_ptr 让编译器自动处理这一切
    """
    # 当前 program 的 ID
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    # ========== 创建 Block Pointer ==========
    # A: (M, K) 行主序 → 取 (BLOCK_M, BLOCK_K) 的 tile
    # 对应 CUDA: gmem_blockA_f4_ptr 的计算
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr,
        shape=(M, K),                        # 整个矩阵形状
        strides=(stride_am, stride_ak),      # 步长
        offsets=(pid_m * BLOCK_M, 0),        # 起始偏移 (行偏移, K偏移=0)
        block_shape=(BLOCK_M, BLOCK_K),      # tile 大小
        order=(1, 0),                        # 行主序: 列维度连续
    )
    
    # B: (K, N) 行主序 → 取 (BLOCK_K, BLOCK_N) 的 tile
    # 对应 CUDA: gmem_blockB_f4_ptr 的计算
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr,
        shape=(K, N),                        # 整个矩阵形状
        strides=(stride_bk, stride_bn),      # 步长
        offsets=(0, pid_n * BLOCK_N),        # 起始偏移 (K偏移=0, 列偏移)
        block_shape=(BLOCK_K, BLOCK_N),      # tile 大小
        order=(1, 0),                        # 行主序
    )
    
    # ========== 初始化 FP32 累加器 ==========
    # 对应 CUDA: float4 reg_c[...]{0};
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # ========== K 维度循环 ==========
    # 对应 CUDA simt_smem: for (int k = 0; k < K; k += BlockTileK)
    for k in range(0, K, BLOCK_K):
        # 加载 A tile (BLOCK_M x BLOCK_K)
        # boundary_check=(0,1) → 自动处理 M 和 K 方向的边界
        # 对应 CUDA: global → smem 的搬运循环 + __syncthreads()
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        
        # 加载 B tile (BLOCK_K x BLOCK_N)
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        
        # 矩阵乘累加
        # 对应 CUDA: 双重循环 reg_c[i*ldm + j] += reg_a[i] * reg_b[j]
        acc = tl.dot(a_tile, b_tile, acc=acc)
        
        # 移动到下一个 K tile
        # 对应 CUDA: k += BlockTileK (循环变量自动推进)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    # ========== 写回结果 ==========
    c = acc.to(tl.float16)
    
    # C 的 Block Pointer
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr,
        shape=(M, N),
        strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N),
        order=(1, 0),
    )
    tl.store(c_block_ptr, c, boundary_check=(0, 1))

In [ ]:
def matmul_block_ptr(a: torch.Tensor, b: torch.Tensor,
                     BLOCK_M=128, BLOCK_N=128, BLOCK_K=32) -> torch.Tensor:
    """Block Pointer GEMM 的 host 端包装函数。"""
    assert a.dtype == torch.float16 and b.dtype == torch.float16
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    matmul_block_ptr_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c

In [ ]:
# ========== 正确性验证 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

c_triton = matmul_block_ptr(a, b)
c_ref = torch.matmul(a, b)

print(f"矩阵规模: A({M}x{K}) @ B({K}x{N}) = C({M}x{N})")
print(f"最大绝对误差: {(c_triton - c_ref).abs().max().item():.4f}")
print(f"平均绝对误差: {(c_triton - c_ref).abs().mean().item():.4f}")
print(f"相对误差 (L2): {torch.norm(c_triton.float() - c_ref.float()) / torch.norm(c_ref.float()):.6f}")
print(f"正确性检查 (atol=1.0): {torch.allclose(c_triton, c_ref, atol=1.0)}")

## 12.5 手动指针 vs Block Pointer 对比

让我们并排比较两种方式，然后做性能对比。

In [ ]:
# ========== 手动指针版本 (来自第09章) ==========
@triton.jit
def matmul_manual_ptr_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """手动指针计算的 GEMM kernel (第09章方式)。"""
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    # 手动计算行列偏移
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    
    # 手动构造 2D 指针矩阵
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k_start in range(0, K, BLOCK_K):
        # 手动计算 mask
        a_tile = tl.load(
            a_ptrs,
            mask=(rm[:, None] < M) & (rk[None, :] + k_start < K),
            other=0.0
        )
        b_tile = tl.load(
            b_ptrs,
            mask=(rk[:, None] + k_start < K) & (rn[None, :] < N),
            other=0.0
        )
        acc = tl.dot(a_tile, b_tile, acc=acc)
        
        # 手动推进指针
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    c = acc.to(tl.float16)
    c_ptrs = c_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [ ]:
# ========== 性能对比: 手动指针 vs Block Pointer ==========
def benchmark_kernel(kernel_fn, M, N, K, BLOCK_M, BLOCK_N, BLOCK_K, num_warmup=10, num_rep=50):
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    c = torch.empty(M, N, device='cuda', dtype=torch.float16)
    
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    for _ in range(num_warmup):
        kernel_fn[grid](
            a, b, c, M, N, K,
            a.stride(0), a.stride(1),
            b.stride(0), b.stride(1),
            c.stride(0), c.stride(1),
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        )
    torch.cuda.synchronize()
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        kernel_fn[grid](
            a, b, c, M, N, K,
            a.stride(0), a.stride(1),
            b.stride(0), b.stride(1),
            c.stride(0), c.stride(1),
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        )
    end.record()
    torch.cuda.synchronize()
    
    elapsed_ms = start.elapsed_time(end) / num_rep
    tflops = 2.0 * M * N * K / (elapsed_ms * 1e-3) / 1e12
    return elapsed_ms, tflops

M, N, K = 2048, 2048, 1024
BM, BN, BK = 128, 128, 32

ms_manual, tflops_manual = benchmark_kernel(matmul_manual_ptr_kernel, M, N, K, BM, BN, BK)
ms_block, tflops_block = benchmark_kernel(matmul_block_ptr_kernel, M, N, K, BM, BN, BK)

print(f"矩阵规模: M={M}, N={N}, K={K}, Block=({BM},{BN},{BK})")
print(f"{'方式':>15} | {'时间(ms)':>10} | {'TFLOPS':>8}")
print("-" * 45)
print(f"{'手动指针':>15} | {ms_manual:>10.3f} | {tflops_manual:>8.2f}")
print(f"{'Block Pointer':>15} | {ms_block:>10.3f} | {tflops_block:>8.2f}")
print(f"\nBlock Pointer 加速比: {ms_manual/ms_block:.2f}x")

## 12.6 Triton 编译器如何映射到 Shared Memory

### 编译器的决策过程

当 Triton 编译器看到如下代码时：

```python
for k in range(0, K, BLOCK_K):
    a_tile = tl.load(a_block_ptr, ...)   # 加载 A tile
    b_tile = tl.load(b_block_ptr, ...)   # 加载 B tile
    acc = tl.dot(a_tile, b_tile, acc=acc) # 矩阵乘
```

编译器会分析：

1. **a_tile 和 b_tile 的生命周期**：它们在 `tl.dot` 中被消费，循环下一轮会重新加载
2. **数据复用模式**：`tl.dot` 的内部实现需要多次访问 a_tile 和 b_tile 的不同行/列
3. **最优存储位置**：
   - 如果 tile 很小 → 可能直接放在寄存器中
   - 如果 tile 较大 → 先放到 Shared Memory，再逐步加载到寄存器

### 编译器生成的等效 CUDA 代码（概念性）

```
Triton 代码:                          编译器生成的等效操作:
                                      
a_tile = tl.load(a_block_ptr)    →    ┌─ cp.async global → smem (异步拷贝)
                                      │  cp.async.commit_group
                                      │  cp.async.wait_group 0
                                      └─ __syncthreads()
                                      
acc = tl.dot(a_tile, b_tile)     →    ┌─ ldmatrix smem → reg  (从 smem 加载到寄存器)
                                      │  mma.sync.aligned    (Tensor Core 运算)
                                      └─ (多轮 ldmatrix + mma)
```

### make_block_ptr 给编译器的额外信息

```
手动指针:                    Block Pointer:
  编译器看到的:               编译器看到的:
  - 一堆分散的指针           - 结构化的 2D 块
  - 不确定访问模式           - 明确的 (M,K) shape
  - 可能需要保守优化          - 明确的 block_shape
                             - 明确的 order (行/列主序)
                             → 可以大胆优化!
```

**关键**：`make_block_ptr` 的 `order` 参数告诉编译器数据的内存布局，
这使得编译器可以：
- 选择最优的全局内存访问模式（合并访问 coalesced access）
- 规划 Shared Memory 的布局以避免 bank conflict
- 使用 `ldmatrix` 指令高效加载到 Tensor Core 的寄存器中

## 12.7 Bank Conflict 概念

### 什么是 Bank Conflict？

Shared Memory 被划分为 **32 个 bank**（与 warp 中的 32 个线程对应）。
每个 bank 宽 4 bytes，连续的 4 bytes 属于不同的 bank。

```
Shared Memory 的 Bank 分布 (每行 4 bytes):

地址:  0x00  0x04  0x08  0x0C  ...  0x7C
Bank:    0     1     2     3   ...    31
地址:  0x80  0x84  0x88  0x8C  ...  0xFC
Bank:    0     1     2     3   ...    31
       ↑ Bank 0 再次出现
```

### 无冲突 vs 有冲突

```
无 Bank Conflict:                     有 Bank Conflict (2-way):
Thread 0 → Bank 0                    Thread 0 → Bank 0 ─┐
Thread 1 → Bank 1                    Thread 1 → Bank 0 ─┤ 冲突!
Thread 2 → Bank 2                    Thread 2 → Bank 2  │ 需要串行化
Thread 3 → Bank 3                    Thread 3 → Bank 2 ─┘
...                                  ...
Thread 31 → Bank 31                  Thread 31 → Bank 30

→ 1 个周期完成                        → 2 个周期完成 (性能减半!)
```

### GEMM 中常见的 Bank Conflict

```
A 矩阵存储在 Shared Memory (行主序, FP16):
列: 0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15
    ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
行0 │B0│B0│B1│B1│B2│B2│B3│B3│B4│B4│B5│B5│B6│B6│B7│B7│ (FP16=2B, 两个FP16=4B=1 bank)
行1 │B0│B0│B1│B1│B2│B2│B3│B3│B4│B4│B5│B5│B6│B6│B7│B7│
    └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

如果多个线程需要读取同一列的不同行 → 可能命中相同 bank → 冲突!
```

### CUDA 中的解决方案

在 CUDA `simt_smemT.cu` 中，A 矩阵在存入 Shared Memory 时进行了**转置**：
```cpp
// simt_smemT.cu: 将 A 的行列互换存储
smem_blockA_hf_ptr[(offset_st_smem_ay * float4_element_num + e) * BlockTileM + offset_st_smem_ax] 
    = reinterpret_cast<half*>(&buffer)[e];
// 效果: 同一列的数据在 smem 中变成同一行 → 读取时不会 bank conflict
```

### Triton 的优势

**Triton 编译器自动处理 bank conflict 避免！**

当你使用 `make_block_ptr` 并指定 `order=(1, 0)` 时：
- 编译器知道数据是行主序的
- 编译器会自动在 Shared Memory 中安排数据布局
- 可能自动 padding 或 swizzle 来避免 bank conflict
- 你不需要（也无法）手动干预

这就是为什么 `order` 参数很重要 —— 它不仅影响 Global Memory 的读取模式，
还影响 Shared Memory 的布局策略。

## 12.8 tl.advance 的工作原理

`tl.advance` 用于移动 block pointer 的偏移，而不改变其他属性：

```python
# 沿 K 方向移动
a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
#                                      ↑      ↑
#                               M方向偏移  K方向偏移
```

```
矩阵 A (M x K):

  K=0    BK    2BK   3BK   4BK
  ┌──────┬──────┬──────┬──────┐
  │tile 0│tile 1│tile 2│tile 3│  ← advance(0, BK) 依次移动
  │      │      │      │      │
  └──────┴──────┴──────┴──────┘

advance(a_block_ptr, (0, BLOCK_K)) 等价于:
  offsets = (offsets[0] + 0, offsets[1] + BLOCK_K)
  即: (pid_m * BM, 0) → (pid_m * BM, BK) → (pid_m * BM, 2BK) → ...
```

### 对比手动指针推进

```python
# 手动方式: 需要了解步长细节
a_ptrs += BLOCK_K * stride_ak    # 容易搞错 stride 方向

# Block Pointer: 语义清晰
a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))  # M方向不动, K方向前进
```

In [ ]:
# ========== 多种矩阵尺寸的性能对比 ==========
print("Block Pointer vs 手动指针: 不同矩阵尺寸")
print(f"{'M':>6} {'N':>6} {'K':>6} | {'手动(ms)':>10} {'Block(ms)':>10} | {'加速比':>8}")
print("-" * 60)

for M, N, K in [(1024, 1024, 512), (2048, 2048, 1024), (4096, 4096, 1024), (2048, 2048, 2048)]:
    BM, BN, BK = 128, 128, 32
    try:
        ms_m, _ = benchmark_kernel(matmul_manual_ptr_kernel, M, N, K, BM, BN, BK)
        ms_b, _ = benchmark_kernel(matmul_block_ptr_kernel, M, N, K, BM, BN, BK)
        print(f"{M:>6} {N:>6} {K:>6} | {ms_m:>10.3f} {ms_b:>10.3f} | {ms_m/ms_b:>7.2f}x")
    except Exception as e:
        print(f"{M:>6} {N:>6} {K:>6} | ERROR: {str(e)[:30]}")

## 12.9 总结

### 本章要点

1. **Shared Memory 的作用**：将 Global Memory 的数据缓存到 on-chip 快速存储中，
   实现 block 内线程的数据共享和复用

2. **CUDA vs Triton 的 Shared Memory 管理**：
   | 方面 | CUDA (simt_smem) | Triton |
   |------|-----------------|--------|
   | 声明 | `__shared__ float4 smem[...]` | 编译器自动 |
   | 搬运 | 手动循环 global→smem | `tl.load` 自动 |
   | 同步 | `__syncthreads()` | 编译器自动 |
   | Bank Conflict | 手动 padding/transpose | 编译器自动 |

3. **make_block_ptr 的优势**：
   - 结构化的 2D 块访问描述
   - `boundary_check` 替代手动 mask
   - `tl.advance` 简化指针推进
   - `order` 参数帮助编译器优化内存布局

4. **Bank Conflict**：
   - Shared Memory 有 32 个 bank
   - 同一 warp 内的线程访问同一 bank 会串行化
   - CUDA 需要手动避免（如 simt_smemT 的转置存储）
   - Triton 编译器自动处理

### 练习

1. **order 参数实验**：将 `order=(1,0)` 改为 `order=(0,1)`，观察性能变化
2. **移除 boundary_check**：当 M,N,K 是 BLOCK 的整数倍时，尝试不使用 `boundary_check`
3. **PTX 分析**：使用 `matmul_block_ptr_kernel.warmup(...)` 查看编译器是否插入了 Shared Memory 操作
4. **思考题**：为什么 `make_block_ptr` 需要 `shape` 参数？（提示：边界检查需要知道矩阵的实际大小）

### 下一章预告

第13章将介绍 **L2 缓存优化 (Swizzle)**，通过调整 program 的发射顺序来提高 L2 cache 命中率。
这对应 CUDA 中 `simt_smemT` 的概念 —— 不仅优化 Shared Memory 的布局，
还优化 block 之间的调度顺序。